This notebook creates training and test datasets from the complete logitudinal dataset created from the previous notebook. 

In [1]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
run_date_str = '20250131'  # queries will pull data up to 1 day prior to run_date_str
s3_output = f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}' # directory to store the raw data

df = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/processed_train.parquet') # read 'processed_train.parquet' generated from the previous notebook

In [3]:
df.shape

(12892153, 81)

In [4]:
# To make training samples similar to the accounts we're going to predict on,
# only keep samples where the restaurants have been on platform for 90 days and don't have 30-day no processing in the past 30 days
df['days_delta_first_obs'] = (
    df['dt'] - df.groupby('rid')['dt'].transform('min')
).dt.days
# df = df.loc[(df['days_delta_first_obs'] >= 30) & (df['days_delta_first_obs'] <= 89) & \
#             (df['noprocessing'] == 0)]

# df = df.loc[(df['days_delta_first_obs'] >= 90) & (df['noprocessing'] == 0)]

In [5]:
horizon = 60  # prediction horizon
# variables to keep
keep_vars = ["rid", "dt", "label_60", "flag_ge50_3d_over_7d",
    "state", "parent_market_segment", "restaurant_type", "account_restaurant_category", "days_delta_first_obs",
    "first_loan_date", "days_with_toast", "months_with_toast",
    "first_loan_date_90d", "has_oo_mod", "has_gc_mod",
    "noprocessing_last_90d", "gpv_mean_90d", "gpv_mean_28d", "gpv_median_28d",
    "gpv_cv_90d", "log_gpv_std_90d", "gpv_median_28d_mean_90d_diff",
    "gpv_median_28d_median_84d_diff", "gpv_median_28d_28ddiff",
    "gpv_median_28d_84ddiff", "gpv_per_hour_median_28d",
    "gpv_per_hour_median_28d_28ddiff", "days_no_gpv_90d",
    "days_no_gpv_28d", "days_no_gpv_28d_28ddiff",
    "tx_hours_mean_14d", "tx_hours_median_14d", "tx_hours_median_28d",
    "tx_hours_median_28d_28ddiff", "live_online_ordering_module_count",
    "live_saas_mrr", "live_saas_module_count", "live_gift_card_module_count",
    "amount_sum", "total_orders", "total_no_of_orders_30d",
    "median_no_of_orders_180d", "std_orders_180d", "gmv_sum_30d",
    "total_size_orders_90", "GMV_90D_Percent_Change",
    "GMV_YoY_90D_Percent_Change", "gpv_mean_365d",
    "gpv_mean_180d", "gpv_cv_180d", "gpv_median_84d",
    "gpv_median_180d", "gpv_median_28d_median_180d_diff",
    "gpv_per_hour", "noprocessing_last_180d",'restaurant_guid', 'payroll_bounce_count_1m',
       'payroll_bounce_fee_sum_1m', 'payroll_bounce_amount_sum_1m',
       'latest_bounce_reason_1m', 'payroll_bounce_count_2m',
       'payroll_bounce_fee_sum_2m', 'payroll_bounce_amount_sum_2m',
       'latest_bounce_reason_2m', 'payroll_bounce_count_3m',
       'payroll_bounce_fee_sum_3m', 'payroll_bounce_amount_sum_3m',
       'latest_bounce_reason_3m', 'payroll_bounce_count_6m',
       'payroll_bounce_fee_sum_6m', 'payroll_bounce_amount_sum_6m',
       'latest_bounce_reason_6m']

In [6]:
from toast_cap.utilities.pd_functions import train_test_split_by_time, non_overlapping_xs

## sample every 14 days
sample_stride = 1
num_test_days = 120  # the last 120 days in the dataset will be used as the test dataset 

# reduce the dataset
df_subset = df.loc[(df['dt'] >= pd.to_datetime('2018-07-01')), keep_vars].copy()

In [7]:
# remove covid data
excl_covid_start = pd.to_datetime('2020-03-10') - pd.to_timedelta(horizon, unit = 'd')
excl_covid_end = pd.to_datetime('2021-03-31')
df_subset = df_subset.loc[(~df_subset['dt'].between(excl_covid_start, excl_covid_end))]

# create train test split and take cross-sectional samples every 14 days
df_train, df_test = train_test_split_by_time(df=df_subset, horizon=horizon, test_buffer_days=num_test_days)
df_train_xs = non_overlapping_xs(df_train, horizon=sample_stride)
df_test_xs = non_overlapping_xs(df_test, horizon=sample_stride)

print(f"training sample date range: {df_train_xs['dt'].min().strftime('%Y-%m-%d')}, {df_train_xs['dt'].max().strftime('%Y-%m-%d')}")
print(f"test sample date range: {df_test_xs['dt'].min().strftime('%Y-%m-%d')}, {df_test_xs['dt'].max().strftime('%Y-%m-%d')}")

df_train_xs.to_parquet(os.path.join(s3_output, 'train_xs.parquet'))
df_test_xs.to_parquet(os.path.join(s3_output, 'test_xs.parquet'))

training sample date range: 2022-01-02, 2024-06-04
test sample date range: 2024-08-04, 2024-12-01


In [8]:
df_train_xs['days_delta_first_obs'].value_counts()

1      17373
2      17366
3      17362
4      17348
5      17341
       ...  
881     5779
882     5762
883     5742
884     5733
885     5727
Name: days_delta_first_obs, Length: 886, dtype: int64

In [9]:
df_train_xs['flag_ge50_3d_over_7d'].value_counts()

1    9897985
0     404753
Name: flag_ge50_3d_over_7d, dtype: int64

In [10]:
df_train_xs['flag_ge50_3d_over_7d'].shape

(10302738,)

In [11]:
# Calculate missing stats for each variable
missing_stats = pd.DataFrame({
    'Total Count': df.count(),  # Non-missing values count
    'Missing Count': df.isnull().sum(),  # Missing values count
    '% Missing': df.isnull().mean() * 100  # Percentage of missing values
})

# Sort by % Missing in descending order
missing_stats = missing_stats.sort_values(by='% Missing', ascending=False)
print(missing_stats)

                              Total Count  Missing Count  % Missing
payroll_bounce_count_1m            162031       12730122  98.743181
payroll_bounce_fee_sum_1m          162031       12730122  98.743181
payroll_bounce_amount_sum_1m       162031       12730122  98.743181
payroll_bounce_count_2m            250385       12641768  98.057850
payroll_bounce_fee_sum_2m          250385       12641768  98.057850
...                                   ...            ...        ...
gpv_date_min                     12892153              0   0.000000
restaurant_guid                  12892153              0   0.000000
latest_bounce_reason_1m          12892153              0   0.000000
latest_bounce_reason_6m          12892153              0   0.000000
days_delta_first_obs             12892153              0   0.000000

[82 rows x 3 columns]


In [12]:
df_train_xs['latest_bounce_reason_1m'].value_counts(dropna=False)

-1     10201742
 0        91338
 3         1970
 1         1618
 5         1428
 10        1154
 6         1017
 2          785
 98         479
 4          387
 9          357
 8          336
 11         127
Name: latest_bounce_reason_1m, dtype: int64